<a href="https://colab.research.google.com/github/NandiniN2215/AILab/blob/main/1BM25CS488_Week7_FOL_Unification_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

class Variable:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name


class Constant:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name


class Predicate:
    def __init__(self, name, *args):
        self.name = name
        self.args = list(args)

    def __repr__(self):
        return self.name + "(" + ", ".join(map(str, self.args)) + ")"


def apply_substitution(term, subs):
    if isinstance(term, Variable):
        while term in subs:
            term = subs[term]
        return term

    if isinstance(term, Predicate):
        return Predicate(
            term.name,
            *[apply_substitution(arg, subs) for arg in term.args]
        )

    return term


def occurs(var, term, subs):
    term = apply_substitution(term, subs)

    if var == term:
        return True

    if isinstance(term, Predicate):
        return any(occurs(var, arg, subs) for arg in term.args)

    return False



def unify(x, y, subs):

    x = apply_substitution(x, subs)
    y = apply_substitution(y, subs)

    # Step 1: Check if both are identical
    if x == y:
        return subs

    # Step 2: If x is a variable
    if isinstance(x, Variable):
        if occurs(x, y, subs):
            return None
        subs[x] = y
        return subs

    # Step 3: If y is a variable
    if isinstance(y, Variable):
        if occurs(y, x, subs):
            return None
        subs[y] = x
        return subs

    # Step 4: Compare constants
    if isinstance(x, Constant) and isinstance(y, Constant):
        if x.name == y.name:
            return subs
        else:
            return None

    # Step 5: Check predicates
    if not isinstance(x, Predicate) or not isinstance(y, Predicate):
        return None

    # Step 6: Check predicate names
    if x.name != y.name:
        return None

    # Step 7: Check number of arguments
    if len(x.args) != len(y.args):
        return None

    # Step 8: Unify corresponding arguments
    for a, b in zip(x.args, y.args):
        subs = unify(a, b, subs)

        if subs is None:
            return None

    return subs


def unification(expr1, expr2):
    print("Sentence 1:", expr1)
    print("Sentence 2:", expr2)

    subs = unify(expr1, expr2, {})

    if subs is None:
        print("\nUnification FAILED!")
    elif not subs:
        print("\nAlready identical. Substitution: NIL")
    else:
        print("\nUnification SUCCESSFUL!")
        print("Substitution set:")

        for var, value in subs.items():
            print(var, "=", apply_substitution(value, subs))


# REAL-WORLD EXAMPLE

# Sentence 1: Rahul likes Cricket
# Sentence 2: Priya likes Cricket

expr1 = Predicate("Likes", Constant("Rahul"), Constant("Cricket"))
expr2 = Predicate("Likes", Constant("Priya"), Constant("Cricket"))

unification(expr1, expr2)


# Example 2: A variable is used
# Sentence 1: Rahul likes X
# Sentence 2: Rahul likes Football

print("\n--- Example 2 ---")

X = Variable("X")

expr1 = Predicate("Likes", Constant("Rahul"), X)
expr2 = Predicate("Likes", Constant("Rahul"), Constant("Football"))

unification(expr1, expr2)


# Example 3: Different relationships

print("\n--- Example 3 ---")

expr1 = Predicate("Likes", Constant("Amit"), Constant("Music"))
expr2 = Predicate("Hates", Constant("Amit"), Constant("Music"))

unification(expr1, expr2)


# Example 4: Student studies a subject

print("\n--- Example 4 ---")

Student = Variable("Student")
Subject = Variable("Subject")

expr1 = Predicate("Studies", Student, Constant("AI"))
expr2 = Predicate("Studies", Constant("Priya"), Subject)

unification(expr1, expr2)


Sentence 1: Likes(Rahul, Cricket)
Sentence 2: Likes(Priya, Cricket)

Unification FAILED!

--- Example 2 ---
Sentence 1: Likes(Rahul, X)
Sentence 2: Likes(Rahul, Football)

Unification SUCCESSFUL!
Substitution set:
X = Football

--- Example 3 ---
Sentence 1: Likes(Amit, Music)
Sentence 2: Hates(Amit, Music)

Unification FAILED!

--- Example 4 ---
Sentence 1: Studies(Student, AI)
Sentence 2: Studies(Priya, Subject)

Unification SUCCESSFUL!
Substitution set:
Student = Priya
Subject = AI
